In [ ]:
import numpy as np
import glob
print('hello')

In [ ]:
files = glob.glob("C:/Users/Rahul/OneDrive/Desktop/Learning/Projects/nyc_taxi/raw/*/*.parquet")
print('Number of files ', len(files))
for f in files:
    print(f)

In [1]:
from pathlib import Path
import polars as pl

PROJECT_ROOT = Path(r"C:/Users/Rahul/OneDrive/Desktop/Learning/Projects/nyc_taxi")
INTERIM_DIR = PROJECT_ROOT / "interim"

files = [
    INTERIM_DIR / "yellow_taxi_2023_normalized.parquet",
    INTERIM_DIR / "yellow_taxi_2024_normalized.parquet",
    INTERIM_DIR / "yellow_taxi_2025_normalized.parquet",
]

lf = pl.scan_parquet([str(f) for f in files])

lf = lf.with_columns([
    pl.col("tpep_pickup_datetime").dt.year().alias("pickup_year"),
    pl.col("tpep_pickup_datetime").dt.month().alias("pickup_month"),
    (
        (pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
        .dt.total_minutes()
    ).alias("trip_duration_minutes"),
])

lf

In [2]:
dropped_fare_lf = lf.filter(pl.col("fare_amount") <= 0)
summary = dropped_fare_lf.select([
    pl.len().alias("row_count"),
    pl.col("fare_amount").min().alias("min_fare"),
    pl.col("fare_amount").max().alias("max_fare"),
    pl.col("total_amount").min().alias("min_total_amount"),
    pl.col("total_amount").max().alias("max_total_amount"),
    pl.col("trip_distance").min().alias("min_trip_distance"),
    pl.col("trip_distance").max().alias("max_trip_distance"),
    pl.col("trip_duration_minutes").min().alias("min_duration_minutes"),
    pl.col("trip_duration_minutes").max().alias("max_duration_minutes"),
]).collect()

print(summary)

shape: (1, 9)
┌───────────┬──────────┬──────────┬────────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ row_count ┆ min_fare ┆ max_fare ┆ min_total_ ┆ … ┆ min_trip_ ┆ max_trip_ ┆ min_durat ┆ max_durat │
│ ---       ┆ ---      ┆ ---      ┆ amount     ┆   ┆ distance  ┆ distance  ┆ ion_minut ┆ ion_minut │
│ u32       ┆ f64      ┆ f64      ┆ ---        ┆   ┆ ---       ┆ ---       ┆ es        ┆ es        │
│           ┆          ┆          ┆ f64        ┆   ┆ f64       ┆ f64       ┆ ---       ┆ ---       │
│           ┆          ┆          ┆            ┆   ┆           ┆           ┆ i64       ┆ i64       │
╞═══════════╪══════════╪══════════╪════════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 3962305   ┆ -2261.2  ┆ 0.0      ┆ -2265.45   ┆ … ┆ 0.0       ┆ 342344.85 ┆ -1177     ┆ 6762      │
└───────────┴──────────┴──────────┴────────────┴───┴───────────┴───────────┴───────────┴───────────┘


In [3]:
categorical_cols = [
    "VendorID",
    "RatecodeID",
    "store_and_fwd_flag",
    "payment_type",
    "source_year",
    "source_month",
]

for col in categorical_cols:
    print(f"\n=== Value counts for {col} in dropped fare rows ===")
    display(
        dropped_fare_lf
        .group_by(col)
        .agg(pl.len().alias("row_count"))
        .sort("row_count", descending=True)
        .collect()
    )


=== Value counts for VendorID in dropped fare rows ===


VendorID,row_count
i64,u32
2,3940645
1,20358
6,1302



=== Value counts for RatecodeID in dropped fare rows ===


RatecodeID,row_count
f64,u32
null,2187971
1.0,1560199
2.0,119572
5.0,50360
3.0,27469
4.0,13933
99.0,2777
6.0,24



=== Value counts for store_and_fwd_flag in dropped fare rows ===


...

payment_type,row_count
i64,u32
0,2187971
4,1083637
2,431536
3,253660
1,5494
5,7



=== Value counts for source_year in dropped fare rows ===


source_year,row_count
i32,u32
2025,2819518
2024,748284
2023,394503



=== Value counts for source_month in dropped fare rows ===


source_month,row_count
i32,u32
11,508457
10,438447
5,423244
6,373072
9,359706
…,…
3,301036
4,278151
2,251407
